# Stylized Neural Painting Demo App

Three types of brushes are available to choose from:
- **Oil Paint**, textured strokes for a classic oil painting feel
- **Watercolor**, soft, blended strokes with watercolor aesthetics (a bit slower)
- **Rectangle**, geometric, blocky strokes for a mosaic-like effect

Two modes:
- **Painting**, recreates the image with brush strokes
- **Style Transfer**, paints the image, then restyle the strokes using a second "style" image

## Quick Setup

If you're running this on **Google Colab**, do the following:
1. **Enable GPU** (strongly recommended): go to **Runtime → Change runtime type** and select **T4 GPU**. Without a GPU, painting will be extremely slow.
2. **Run both code cells below** in order — click the play ▶ button on each, or use **Runtime → Run all**.
3. An interactive app will appear at the bottom after both cells finished running. Upload an image, pick a brush, and hit **Paint**. You can also play with style transfer mode.

> **Troubleshooting:** As part of the env set up, I install only the `kornia` package because other dependencies come pre-installed in Google Colab. This is much faster than setting up new env + there is
 guaranteed compatibility with the underlying CUDA version.
> However, if you get import errors, version conflicts, etc., you can try to tweak dependencies based
 on the versions found in the uv.lock, which is part of the cloned repository (first cell).

This notebook works also outside the Colab and can be run locally via jupyter lab after [setting up your env](https://github.com/Druudik/stylized-neural-painting?tab=readme-ov-file#local-setup) (just comment the first notebook cell so that you don't clone the repo again).

## Read Before Running on Colab!

- **Public link:** In **Colab**, the app (which uses [gradio](https://github.com/gradio-app/gradio) package) automatically creates a public link from which you can open the app on the separate browser tab. This connects directly to your running notebook kernel. **It is safe by default**, but you **should not mount Google Drive or paste any secrets** into the notebook while the link is active, especially, if you're also modifying `allowed_paths` parameter of the `demo.launch(...)`.
- **When you're done:** Go to **Runtime → Disconnect and delete runtime**. This frees your free Colab GPU quota and **disables the public link**.

## Notes

Good **style image** is rather abstract, with specific shapes, textures and colors. Some examples: *[Composition with Red, Blue and Yellow](https://en.wikipedia.org/wiki/Composition_with_Red,_Blue_and_Yellow)*, *[Simultaneous Contrasts](https://en.wikipedia.org/wiki/Simultaneous_Contrasts)* or *[The Scream](https://en.wikipedia.org/wiki/The_Scream)*.

Regarding **performance**:
* Different brushes have different painting schedules (number of the brush strokes and grid splitting schedule) and rendering speeds. Oil paint is reasonably fast and have a painting schedule that fits most of the images.
* The Watercolor brush takes a bit longer during final rendering after optimization (up to 60 seconds with resolution set to 1024).
* Higher resolutions are slower in general.

This notebook is part of [my reimplementation](https://github.com/Druudik/stylized-neural-painting) of the [Stylized Neural Painting](https://jiupinjia.github.io/neuralpainter/) paper. It's licensed under [MIT license](https://github.com/Druudik/stylized-neural-painting/blob/main/LICENSE).

In [ ]:
#@title **Click to Run Setup** { run: "auto", display-mode: "form" }

# If running things locally, you can comment following lines if you have repo and env already set up.
!git clone https://github.com/Druudik/stylized-neural-painting.git
%cd stylized-neural-painting
%pip install -q kornia

In [ ]:
#@title **Click to Start the App** { run: "auto", display-mode: "form" }

import os
import sys
sys.path.append(".")

import io
import urllib.request
from dataclasses import dataclass
from collections.abc import Iterable

import gradio as gr
import torch
import torchvision.transforms.functional as TF
from PIL import Image
from torch import Tensor
from huggingface_hub import hf_hub_download

from painting.brushes import (
    Brush,
    DifferentiableBrush,
    GeneralNeuralRenderer,
    RectangleBrush,
    TextureBrush,
    WatercolorBrush,
    ColorMaskNeuralRenderer,
)
from painting.loss import PixelLoss
from painting.painter import Painter
from painting.samplers import TargetGuidedSampler
from painting.style_transferer import StyleTransferer


# ----- Constants -----

IS_COLAB = "COLAB_RELEASE_TAG" in os.environ
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
GRID_BATCH_SIZE = 50
RENDERING_BATCH_SIZE = 35
STYLE_TRANSFER_RENDERING_BATCH_SIZE = 50
BOUNDARY_OFFSET = 0.1
MIN_LOCAL_PARAM_SIZE = 0.1
MAX_LOCAL_PARAM_SIZE = 0.9

PAINTING_SCHEDULES = {
    "Oil Paint": {},
    "Watercolor": {
        "n_grids_per_dim_schedule": [1, 4, 5],
        "n_strokes_per_grid_schedule": [25, 5, 25],
        "active_set_size_schedule": [25, 5, 25],
        "total_optim_steps_per_active_set_schedule": [400, 200, 400],
    },
    "Rectangle": {
        "n_grids_per_dim_schedule": [1, 2, 4],
        "n_strokes_per_grid_schedule": [50, 5, 2],
        "active_set_size_schedule": [25, 5, 2],
        "total_optim_steps_per_active_set_schedule": [650, 250, 100],
    },
}

STYLE_TRANSFER_PAINTING_SCHEDULES = {
    "Oil Paint": {
        "n_grids_per_dim_schedule": [5],
        "n_strokes_per_grid_schedule": [40],
        "active_set_size_schedule": [20],
        "total_optim_steps_per_active_set_schedule": [240],
    },
    "Watercolor": {
        "n_grids_per_dim_schedule": [5],
        "n_strokes_per_grid_schedule": [25],
        "active_set_size_schedule": [25],
        "total_optim_steps_per_active_set_schedule": [400],
    },
    "Rectangle": {
        "n_grids_per_dim_schedule": [1],
        "n_strokes_per_grid_schedule": [125],
        "active_set_size_schedule": [25],
        "total_optim_steps_per_active_set_schedule": [350],
    },
}

STYLE_TRANSFER_CONFIGS = {
    "Oil Paint": {"color_only_mode": False},
    "Watercolor": {"color_only_mode": False},
    "Rectangle": {"color_only_mode": True},
}

# ----- Helpers -----

@dataclass
class BrushMetadata:
    params_count: int
    pos_indices: list[tuple[int, int]]
    size_indices: list[int]
    color_indices: list[tuple[int, int, int]]


def load_texture(path: str) -> Tensor:
    img = Image.open(path).convert("L")
    return (TF.pil_to_tensor(img).float() / 255.0).to(DEVICE)


def resolve_image(file, url):
    """Load an image from a file upload or a URL."""
    if file is not None:
        path = file.name if hasattr(file, "name") else file
        try:
            return Image.open(path).convert("RGB")
        except Exception as e:
            raise gr.Error(f"Could not open uploaded file: {e}")
    if url and url.strip():
        try:
            with urllib.request.urlopen(url.strip(), timeout=30) as resp:
                data = resp.read()
            return Image.open(io.BytesIO(data)).convert("RGB")
        except Exception as e:
            raise gr.Error(f"Could not load image from URL: {e}")
    raise gr.Error("Please upload an image or paste a URL.")


def create_brush(brush_type: str, canvas_size: int) -> tuple[Brush, BrushMetadata]:
    if brush_type == "Watercolor":
        brush = WatercolorBrush(canvas_size=canvas_size, apply_closing=True, device=DEVICE)
        meta = BrushMetadata(
            params_count=brush.brush_params_count,
            pos_indices=[(0, 1), (2, 3), (4, 5)],
            size_indices=[6],
            color_indices=[(7, 8, 9)],
        )
    elif brush_type == "Oil Paint":
        brush = TextureBrush(
            large_vertical=load_texture("./assets/textures/large_vertical_brush.png"),
            large_horizontal=load_texture("./assets/textures/large_horizontal_brush.png"),
            small_vertical=load_texture("./assets/textures/small_vertical_brush.png"),
            small_horizontal=load_texture("./assets/textures/small_horizontal_brush.png"),
            canvas_size=canvas_size, apply_closing=True, device=DEVICE,
        )
        meta = BrushMetadata(
            params_count=brush.brush_params_count,
            pos_indices=[(0, 1)],
            size_indices=[2, 3],
            color_indices=[(5, 6, 7), (8, 9, 10)],
        )
    elif brush_type == "Rectangle":
        brush = RectangleBrush(canvas_size=canvas_size, device=DEVICE)
        meta = BrushMetadata(
            params_count=brush.brush_params_count,
            pos_indices=[(0, 1)],
            size_indices=[2, 3],
            color_indices=[(4, 5, 6)],
        )
    else:
        raise ValueError(f"Unknown brush type: {brush_type}")
    return brush, meta


def create_diff_brush(brush: Brush) -> DifferentiableBrush:
    repo_map = {
        WatercolorBrush: "Druudik1/watercolor-brush",
        TextureBrush: "Druudik1/texture-brush",
        RectangleBrush: "Druudik1/rectangle-brush",
    }
    repo_id = repo_map[type(brush)]

    if isinstance(brush, WatercolorBrush):
        diff_brush = ColorMaskNeuralRenderer(
            brush_params_count=brush.brush_params_count,
            color_params_indices=[7, 8, 9],
            canvas_size=128, device=DEVICE,
        )
    else:
        diff_brush = GeneralNeuralRenderer(
            brush_params_count=brush.brush_params_count,
            canvas_size=128, device=DEVICE,
        )

    weights = hf_hub_download(repo_id=repo_id, filename="neural_renderer.pt")
    diff_brush.load_state_dict(torch.load(weights, map_location=DEVICE, weights_only=True))
    diff_brush.eval()
    return diff_brush


def pil_to_tensor(img: Image.Image) -> Tensor:
    return TF.pil_to_tensor(img.convert("RGB")).float().to(DEVICE)


def tensor_to_pil(tensor: Tensor) -> Image.Image:
    return TF.to_pil_image(tensor.clamp(0, 255).byte().cpu())


def tqdm_wrapper(iterable: Iterable, desc: str):
    items = list(iterable)
    if len(items) == 1:
        return items
    from tqdm.auto import tqdm
    return tqdm(items, desc=desc, leave=False)


# ----- Painting logic -----

def run_painting(
    image_file,
    image_url,
    brush_type,
    resolution,
    mode,
    style_image,
    style_url,
    progress=gr.Progress(track_tqdm=True)
):
    try:
        target_pil = resolve_image(image_file, image_url)
        canvas_size = int(resolution)
        enable_style = mode == "Style Transfer"

        if enable_style:
            if style_image is None and (not style_url or not style_url.strip()):
                raise gr.Error("Style transfer is enabled but no style image was provided.")
            style_pil = resolve_image(style_image, style_url)

        brush, meta = create_brush(brush_type, canvas_size)
        diff_brush = create_diff_brush(brush)
        initial_canvas = torch.zeros((3, canvas_size, canvas_size), device=DEVICE)

        sampler = TargetGuidedSampler(
            brush_params_count=brush.brush_params_count,
            pos_param_indices=meta.pos_indices,
            size_param_indices=meta.size_indices,
            color_param_indices=meta.color_indices,
            boundary_offset=BOUNDARY_OFFSET,
            device=DEVICE,
        )

        painter = Painter(
            brush=brush,
            brush_pos_indices=meta.pos_indices,
            brush_size_indices=meta.size_indices,
            differentiable_brush_imitator=diff_brush,
            brush_param_sampler=sampler,
            loss=PixelLoss(p=1),
            boundary_offset=BOUNDARY_OFFSET,
            min_local_param_size=MIN_LOCAL_PARAM_SIZE,
            max_local_param_size=MAX_LOCAL_PARAM_SIZE,
            grid_batch_size=GRID_BATCH_SIZE,
            rendering_batch_size=RENDERING_BATCH_SIZE,
            device=DEVICE,
        )

        target_tensor = pil_to_tensor(target_pil)

        # Select painting schedule based on mode.
        if enable_style:
            paint_schedule = STYLE_TRANSFER_PAINTING_SCHEDULES[brush_type]
            closing = True
        else:
            paint_schedule = PAINTING_SCHEDULES.get(brush_type, {})
            closing = False

        # Paint.
        try:
            canvas, brush_params = painter.paint(
                target_image=target_tensor,
                initial_canvas=initial_canvas,
                apply_brush_stroke_closing_during_optim=closing,
                iter_progress_wrapper=tqdm_wrapper,
                **paint_schedule,
            )
        except RuntimeError as e:
            if "out of memory" in str(e).lower():
                torch.cuda.empty_cache()
                return None, "**Error:** Out of GPU memory."
            raise

        painted_pil = tensor_to_pil(canvas)
        status = "Painting completed."

        # Style transfer logic.
        if enable_style:
            st_config = STYLE_TRANSFER_CONFIGS[brush_type]
            style_transferer = StyleTransferer(
                differentiable_brush=diff_brush,
                brush_pos_indices=meta.pos_indices,
                brush_size_indices=meta.size_indices,
                brush_color_indices=meta.color_indices,
                color_only_mode=st_config["color_only_mode"],
                boundary_offset=BOUNDARY_OFFSET,
                min_local_param_size=MIN_LOCAL_PARAM_SIZE,
                max_local_param_size=MAX_LOCAL_PARAM_SIZE,
                rendering_batch_size=STYLE_TRANSFER_RENDERING_BATCH_SIZE,
                device=DEVICE,
            )

            style_tensor = pil_to_tensor(style_pil)
            n_strokes = brush_params.shape[0]

            try:
                stylized_params = style_transferer.transfer(
                    brush_params=brush_params,
                    target_image=target_tensor,
                    style_image=style_tensor,
                    n_grids_per_dim=paint_schedule["n_grids_per_dim_schedule"][0],
                    initial_canvas=initial_canvas,
                    apply_brush_stroke_closing=True,
                    iter_progress_wrapper=tqdm_wrapper,
                )
            except RuntimeError as e:
                if "out of memory" in str(e).lower():
                    torch.cuda.empty_cache()
                    return painted_pil, f"Painting done ({n_strokes:,} strokes) but style transfer ran out of memory. Try a smaller resolution."
                raise

            progress(0.99, desc="Painting stylized brush strokes...")

            stylized_canvas = brush.draw_on_single_canvas(
                brush_params=stylized_params,
                canvas=initial_canvas,
                rendering_batch_size=RENDERING_BATCH_SIZE,
            )
            painted_pil = tensor_to_pil(stylized_canvas)
            status += " Style transfer completed."

        return painted_pil, status

    except gr.Error as e:
        return None, f"**Error:** {e}"
    except Exception as exc:
        return None, f"**Error:** {exc}"
    finally:
        if DEVICE == "cuda":
            torch.cuda.empty_cache()


# ----- Gradio app. -----

def build_app_description() -> str:
    if DEVICE == "cuda":
        device_label = "GPU"
    else:
        device_label = "CPU (slow — enable GPU in Runtime > Change runtime type)"

    parts = [f"# Stylized Neural Painting\n**Device:** {device_label}"]

    if DEVICE != "cuda":
        warning = "> **No GPU detected, painting will be very slow.**"
        if IS_COLAB:
            warning += "\nEnable GPU via **Runtime → Change runtime type → T4 GPU** and restart the notebook."
        parts.append(warning)

    if IS_COLAB:
        parts.append(
            "**You can also open the public link shown above to use this app in a separate browser tab.**\n\n"
            "**DO NOT FORGET TO `Runtime → Disconnect and delete runtime` when you're done with all the painting. "
            "This prevents spending your free GPU units and also disables the public link right away, instead of waiting for the Colab 90 minutes idle timeout or 12h session limit.**"
        )

    return "\n\n".join(parts)


def image_picker(accordion_label: str, **accordion_kwargs) -> tuple[gr.Accordion, gr.File, gr.Textbox]:
    """Create an image picker with file upload and URL input tabs inside an accordion."""
    with gr.Accordion(accordion_label, open=True, **accordion_kwargs) as section:
        with gr.Tabs():
            with gr.Tab("Upload"):
                file_input = gr.File(file_types=["image"], height=120)
            with gr.Tab("Paste URL"):
                url_input = gr.Textbox(
                    placeholder="https://example.com/image.jpg", label="Image URL", lines=1,
                )
    return section, file_input, url_input


def clear_file_on_url_change(url):
    return None if url and url.strip() else gr.update()


def clear_url_on_file_change(file):
    return "" if file is not None else gr.update()


def link_image_inputs(file_input: gr.File, url_input: gr.Textbox):
    url_input.change(fn=clear_file_on_url_change, inputs=[url_input], outputs=[file_input])
    file_input.change(fn=clear_url_on_file_change, inputs=[file_input], outputs=[url_input])


with gr.Blocks() as demo:
    gr.Markdown(build_app_description())

    with gr.Row():
        # Left column: controls.
        with gr.Column(scale=1):
            _, image_file, image_url = image_picker("Pick image to paint")
            brush = gr.Radio(["Oil Paint", "Watercolor", "Rectangle"], value="Oil Paint", label="Brush")
            resolution = gr.Radio(["256", "512", "1024"], value="512", label="Resolution")
            mode = gr.Radio(["Painting", "Style Transfer"], value="Painting", label="Mode")
            style_section, style_file, style_url = image_picker(
                "Pick image for style transfer", visible=False,
            )
            paint_btn = gr.Button("Paint", variant="primary")

        # Right column: output.
        with gr.Column(scale=3):
            result_image = gr.Image(label="Result", show_label=False, format="jpeg")
            status_text = gr.Markdown("")

    # Toggle style image picker visibility when mode changes.
    mode.change(
        fn=lambda m: gr.update(visible=(m == "Style Transfer")),
        inputs=[mode],
        outputs=[style_section],
    )

    # Filling a URL clears the file upload and vice versa.
    link_image_inputs(image_file, image_url)
    link_image_inputs(style_file, style_url)

    # Paint button: lock inputs → run painting → unlock inputs.
    all_inputs = [image_file, image_url, brush, resolution, mode, style_file, style_url, paint_btn]

    paint_btn.click(
        fn=lambda: [gr.update(interactive=False)] * len(all_inputs) + ["*Painting...*"],
        outputs=all_inputs + [status_text],
    ).then(
        fn=run_painting,
        inputs=[image_file, image_url, brush, resolution, mode, style_file, style_url],
        outputs=[result_image, status_text],
        show_progress_on=[result_image],
    ).then(
        fn=lambda: [gr.update(interactive=True)] * len(all_inputs),
        outputs=all_inputs,
    )

demo.launch(height=900, inline=True, quiet=True)